# 02 â€” Logistic Regression Baseline

A serious project starts with a simple baseline. If logistic regression gets within a couple of percentage points of the fancy model, the fancy model is mostly noise.

## Setup and data prep

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

FEATURES = [
    'loan_amnt', 'term', 'int_rate', 'installment', 'grade',
    'emp_length', 'home_ownership', 'annual_inc', 'verification_status',
    'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'fico_range_low',
    'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
    'total_acc',
]
CATEGORICAL = ['term', 'grade', 'home_ownership', 'verification_status', 'purpose', 'addr_state']

def parse_term(s):
    if isinstance(s, str):
        return int(s.strip().split()[0])
    return np.nan

def parse_emp_length(s):
    if not isinstance(s, str):
        return np.nan
    s = s.strip()
    if '<' in s:
        return 0
    if '+' in s:
        return 10
    parts = s.split()
    return int(parts[0]) if parts and parts[0].isdigit() else np.nan

def parse_pct(s):
    if isinstance(s, str):
        s = s.replace('%', '').strip()
        return float(s) if s else np.nan
    return s

# Load and clean
df = pd.read_csv('../data/loan.csv', low_memory=False)
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
df['target'] = (df['loan_status'] == 'Charged Off').astype(int)
df, _ = train_test_split(df, train_size=0.10, stratify=df['target'], random_state=42)

df['term'] = df['term'].map(parse_term)
df['int_rate'] = df['int_rate'].map(parse_pct)
df['revol_util'] = df['revol_util'].map(parse_pct)
df['emp_length'] = df['emp_length'].map(parse_emp_length)

# Data-quality fixes (per notebook 01 findings)
df.loc[df['dti'] > 50, 'dti'] = np.nan
df.loc[df['revol_util'] > 100, 'revol_util'] = np.nan

X = df[FEATURES].copy()
for col in CATEGORICAL:
    X[col] = X[col].astype('category')
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}  Default rate: {y_train.mean():.1%}')


Train: 107,624  Test: 26,907  Default rate: 20.0%


## Fit logistic regression with proper preprocessing

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import time

numeric_features = [f for f in FEATURES if f not in CATEGORICAL]
categorical_features = CATEGORICAL

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), numeric_features),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), categorical_features),
])

# Need to convert categoricals back to object for OneHotEncoder
X_train_lr = X_train.copy()
X_test_lr = X_test.copy()
for col in CATEGORICAL:
    X_train_lr[col] = X_train_lr[col].astype('object')
    X_test_lr[col] = X_test_lr[col].astype('object')

t0 = time.time()
lr_pipeline = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)),
])
lr_pipeline.fit(X_train_lr, y_train)
train_time = time.time() - t0

proba = lr_pipeline.predict_proba(X_test_lr)[:, 1]
print(f'Logistic Regression baseline:')
print(f'  AUC:    {roc_auc_score(y_test, proba):.4f}')
print(f'  PR-AUC: {average_precision_score(y_test, proba):.4f}')
print(f'  Brier:  {brier_score_loss(y_test, proba):.4f}')
print(f'  Train time: {train_time:.1f}s')


Logistic Regression baseline:
  AUC:    0.7116
  PR-AUC: 0.3772
  Brier:  0.2173
  Train time: 0.7s


**Observations**

- Logistic regression with proper preprocessing (median imputation, scaling, one-hot, class-weight balancing) gets **AUC 0.7116** in under a second of training.
- Brier score 0.2173 is high. Class-weight balancing is great for AUC ranking but blows up calibration — the model now treats positives as if they were 50% of the population, so its predicted probabilities are systematically too high. If we needed calibrated probabilities from this model we'd retrain without `class_weight='balanced'`.
- PR-AUC 0.3772 on a 20% positive rate (random would be 0.20) — the model lifts ~1.9x over random in precision-at-recall ordering.


## Bottom line

**Baseline AUC: 0.7116.** Anything we build later has to beat this. If LightGBM only gets +0.005 AUC, we should probably ship the simpler model. (Spoiler from notebook 03: it gets +0.002. The extra complexity is barely earning its keep.)
